# T04 — Preprocessing and feature-engineering contract

**Authorized scope:** fit and verify the frozen no-op preprocessing contract on train rows only; apply it to validation. Held-out row identities are never requested — this notebook does not import the held-out access guard's authorized path at all.

In [1]:
from __future__ import annotations

import hashlib
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


def _find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'data.py').is_file():
            return candidate
    raise RuntimeError(f'Could not locate repository root above {start}')


WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = _find_repo_root(WORKING_DIRECTORY)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import (
    FEATURE_COLUMNS, PROCESSED_COLUMNS,
    finalize_artifact_manifest, load_selector, materialize_pandas,
    open_processed_dataset, sha256_file, write_json_new,
)
from src.audit import AuditContractError, validate_pre_split_frame
from src.split import SplitContractError, membership_hash
from src.preprocessing import IdentityFeatureTransform, preprocessing_contract

NOTEBOOK_PATH = REPO_ROOT / 'notebooks' / 'internal' / 't04_preprocessing.ipynb'
T03_CONFIG_PATH = REPO_ROOT / 'configs' / 't03_audit.json'
T05_CONFIG_PATH = REPO_ROOT / 'configs' / 't05_split.json'
T04_CONFIG_PATH = REPO_ROOT / 'configs' / 't04_preprocessing.json'
SELECTOR_PATH = REPO_ROOT / 'configs' / 'data_manifest.json'
STAGE = 't04_preprocessing'
POPULATION = 'train_validation_development_rows'


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def notebook_source_sha256(path):
    payload = json.loads(path.read_text(encoding='utf-8'))
    source_only = [{'cell_type': c['cell_type'], 'source': ''.join(c.get('source', []))} for c in payload['cells']]
    return hashlib.sha256(json.dumps(source_only, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

## 1. Verify predecessors

In [2]:
t03_config = json.loads(T03_CONFIG_PATH.read_text(encoding='utf-8'))
t05_config = json.loads(T05_CONFIG_PATH.read_text(encoding='utf-8'))
t04_config = json.loads(T04_CONFIG_PATH.read_text(encoding='utf-8'))

required_t03_state = t04_config['input']['predecessor']['required_lifecycle_state']
if t03_config['lifecycle_state'] != required_t03_state:
    raise SplitContractError(f"T03 is not {required_t03_state} (found {t03_config['lifecycle_state']!r})")
required_t05_state = t04_config['input']['split']['required_lifecycle_state']
if t05_config['lifecycle_state'] != required_t05_state:
    raise SplitContractError(f"T05 is not {required_t05_state} (found {t05_config['lifecycle_state']!r})")

for guard, expected in t04_config['scope_guards'].items():
    assert expected is False, f'Scope guard {guard} must be False at notebook start'

t05_run_id = t05_config['lifecycle_state_evidence']['authorizing_run_id']
t05_manifest_path = REPO_ROOT / t05_config['lifecycle_state_evidence']['authorizing_run_manifest']
t05_run_root = t05_manifest_path.parent.parent
membership = pd.read_csv(t05_run_root / 'audit' / 'split_membership.csv')
observed_hash = membership_hash(membership)
expected_hash = t05_config['lifecycle_state_evidence']['membership_sha256']
if observed_hash != expected_hash:
    raise SplitContractError(f'Split membership hash mismatch: expected {expected_hash}, observed {observed_hash}')

print(f'T03: {t03_config["lifecycle_state"]}; T05: {t05_config["lifecycle_state"]} (run {t05_run_id})')

T03: T03_DEVELOPMENT_INTEGRITY_COMPLETE; T05: T05_SPLIT_ACCEPTED (run t05_split_20260818T073132Z_534290)


## 2. Load train and validation rows only

Held-out `_source_row_id`s are never selected out of `membership` here — only `train` and `validation` are read.

In [3]:
selector = load_selector(SELECTOR_PATH, REPO_ROOT)
dataset = open_processed_dataset(selector)
processed_sha256 = selector.payload['processed_sha256']

frame = materialize_pandas(dataset, columns=PROCESSED_COLUMNS, row_limit=None)
if tuple(frame.columns) != PROCESSED_COLUMNS:
    raise AuditContractError('Processed columns changed during materialization')
_ = validate_pre_split_frame(frame, require_zero_based_complete=True)

development_ids = membership.loc[membership['split'].isin(['train', 'validation']), '_source_row_id']
train_ids = membership.loc[membership['split'] == 'train', '_source_row_id']
validation_ids = membership.loc[membership['split'] == 'validation', '_source_row_id']

train_frame = frame.loc[frame['_source_row_id'].isin(train_ids)]
validation_frame = frame.loc[frame['_source_row_id'].isin(validation_ids)]
print(f'train={len(train_frame):,} rows, validation={len(validation_frame):,} rows')

train=9,785,714 rows, validation=2,096,938 rows


## 3. Immutable T04 run initialization

In [4]:
RUN_CREATED_AT = utc_now()
RUN_ID = datetime.now(timezone.utc).strftime('t04_preprocessing_%Y%m%dT%H%M%SZ_%f')
RUN_ROOT = REPO_ROOT / 'outputs' / 'runs' / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)

try:
    git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
    git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
except (OSError, subprocess.CalledProcessError):
    git_head, git_dirty = None, None

contract = preprocessing_contract()
contract_sha256 = hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()
run_config = {
    'run_id': RUN_ID, 'created_at_utc': RUN_CREATED_AT, 'stage': STAGE,
    'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH),
    'src_preprocessing_sha256': sha256_file(REPO_ROOT / 'src' / 'preprocessing.py'),
    'processed_sha256': processed_sha256,
    'authorizing_t05_run_id': t05_run_id,
    'contract_sha256': contract_sha256,
    'git_head': git_head, 'git_dirty': git_dirty,
}
write_json_new(RUN_ROOT, 'audit/run_config.json', run_config)
write_json_new(RUN_ROOT, 'audit/preprocessing_contract.json', {**contract, 'contract_sha256': contract_sha256})
print(f'Run: {RUN_ID}')
print(f'Contract hash: {contract_sha256}')

Run: t04_preprocessing_20260818T080330Z_043364
Contract hash: 67e669c9ed2b81f81b22af4675529cbb1ecc63883806542546c9fec013cd75fc


## 4. Fit on train rows only; apply unchanged to validation

`fit` is called with `train_frame` alone. `validation_frame` is passed only to `transform`, which reads no state derived from calling it -- proving the fit boundary holds on the real data, not just in the unit tests.

In [5]:
transform = IdentityFeatureTransform()
transform.fit(train_frame)

train_output = transform.transform(train_frame)
validation_output = transform.transform(validation_frame)

assert tuple(train_output.columns) == FEATURE_COLUMNS
assert tuple(validation_output.columns) == FEATURE_COLUMNS
assert all(str(dtype) == 'float64' for dtype in train_output.dtypes)
assert all(str(dtype) == 'float64' for dtype in validation_output.dtypes)
assert len(train_output) == len(train_frame)
assert len(validation_output) == len(validation_frame)

# Determinism: transforming the same frame twice gives identical output.
repeat = transform.transform(train_frame)
pd.testing.assert_frame_equal(train_output.reset_index(drop=True), repeat.reset_index(drop=True))

# Missing-value passthrough on whatever is actually present (T03-A found zero, verified again here).
train_missing = int(train_output.isna().sum().sum())
validation_missing = int(validation_output.isna().sum().sum())
print(f'train rows={len(train_output):,}, missing values={train_missing}')
print(f'validation rows={len(validation_output):,}, missing values={validation_missing}')

train rows=9,785,714, missing values=0
validation rows=2,096,938, missing values=0


## 5. Close the run and record status

In [6]:
summary = {
    'run_id': RUN_ID, 'status': 'COMPLETED_T04_PREPROCESSING_CONTRACT',
    'contract_sha256': contract_sha256,
    'train_rows': int(len(train_frame)), 'validation_rows': int(len(validation_frame)),
    'train_missing_values': train_missing, 'validation_missing_values': validation_missing,
    'fit_on_validation_or_held_out': False,
    'held_out_accessed': False,
    'column_order_verified': True, 'dtype_verified': True, 'deterministic_verified': True,
}
write_json_new(RUN_ROOT, 'audit/t04_summary.json', summary)

finalize_artifact_manifest(
    RUN_ROOT, run_id=RUN_ID, final_status=summary['status'], created_at_utc=utc_now(),
    stage=STAGE, population=POPULATION,
    external_artifacts=[
        {'path': selector.processed_path.name, 'role': 'manifest_selected_processed_derivative', 'sha256': processed_sha256, 'status': 'PASS'},
        {'path': 'notebooks/internal/t04_preprocessing.ipynb#sources', 'role': 'human_readable_protocol_source', 'sha256': notebook_source_sha256(NOTEBOOK_PATH), 'status': 'PASS'},
        {'path': 'src/preprocessing.py', 'role': 'reusable_t04_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'preprocessing.py'), 'status': 'PASS'},
    ],
)
summary

{'run_id': 't04_preprocessing_20260818T080330Z_043364',
 'status': 'COMPLETED_T04_PREPROCESSING_CONTRACT',
 'contract_sha256': '67e669c9ed2b81f81b22af4675529cbb1ecc63883806542546c9fec013cd75fc',
 'train_rows': 9785714,
 'validation_rows': 2096938,
 'train_missing_values': 0,
 'validation_missing_values': 0,
 'fit_on_validation_or_held_out': False,
 'held_out_accessed': False,
 'column_order_verified': True,
 'dtype_verified': True,
 'deterministic_verified': True}